In [1]:
import os
os.chdir(r'C:\Users\Admin\Desktop\CUSTOMER COMPLAINT CLASSIFICATION')
print(os.getcwd())

C:\Users\Admin\Desktop\CUSTOMER COMPLAINT CLASSIFICATION


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
import joblib
import json
import os
from datetime import datetime

# Load preprocessed data
df = pd.read_csv('data/processed/complaints_preprocessed.csv')

# Split features and target
X = df['clean_text']
y = df['label']

# Train / eval / test split (70 / 15 / 15) stratified
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_eval, X_test, y_eval, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print('Train size:', len(X_train))
print('Eval size:', len(X_eval))
print('Test size:', len(X_test))

# Save splits to processed folder
X_train.to_csv('data/processed/X_train.csv', index=False)
X_eval.to_csv('data/processed/X_eval.csv', index=False)
X_test.to_csv('data/processed/X_test.csv', index=False)
y_train.to_csv('data/processed/y_train.csv', index=False)
y_eval.to_csv('data/processed/y_eval.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

print('Splits saved successfully')

Train size: 171659
Eval size: 36784
Test size: 36785
Splits saved successfully


In [3]:
from sklearn.metrics import classification_report, f1_score

# Define models
models = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'naive_bayes': MultinomialNB(),
    'linear_svm': LinearSVC(random_state=42, max_iter=2000)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    
    # Build pipeline: TF-IDF + model
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1, 2))),
        ('clf', model)
    ])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Evaluate on eval set
    y_pred = pipeline.predict(X_eval)
    f1 = f1_score(y_eval, y_pred, average='weighted')
    report = classification_report(y_eval, y_pred)
    
    print(f'{name} weighted F1: {f1:.4f}')
    print(report)
    
    # Save artifact
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    artifact_dir = f'models_artifacts/{name}_champion_{timestamp}'
    os.makedirs(artifact_dir, exist_ok=True)
    
    joblib.dump(pipeline, f'{artifact_dir}/model.pkl')
    
    with open(f'{artifact_dir}/metrics.json', 'w') as f:
        json.dump({'model': name, 'weighted_f1': f1}, f, indent=2)
    
    with open(f'{artifact_dir}/classification_report.txt', 'w') as f:
        f.write(report)
    
    results[name] = f1
    print(f'Artifact saved: {artifact_dir}\n')

# Pick champion
champion = max(results, key=results.get)
print('Champion model:', champion)
print('Results:', results)

Training logistic_regression...
logistic_regression weighted F1: 0.8983
                                                    precision    recall  f1-score   support

                       Checking or savings account       0.87      0.93      0.90     15063
Money transfer, virtual currency, or money service       0.87      0.79      0.83      8750
                                          Mortgage       0.97      0.94      0.96      5485
                                      Student loan       0.97      0.94      0.96      3685
                             Vehicle loan or lease       0.92      0.92      0.92      3801

                                          accuracy                           0.90     36784
                                         macro avg       0.92      0.91      0.91     36784
                                      weighted avg       0.90      0.90      0.90     36784

Artifact saved: models_artifacts/logistic_regression_champion_20260324_214228

Training naive_bay

In [4]:
import shutil

# Copy champion model to models/latest/
os.makedirs('models/latest', exist_ok=True)

# Find the logistic regression artifact folder
champion_dir = [d for d in os.listdir('models_artifacts') if d.startswith('logistic_regression_champion')][0]
shutil.copy(f'models_artifacts/{champion_dir}/model.pkl', 'models/latest/model.pkl')

print(f'Champion model copied from {champion_dir} to models/latest/')

Champion model copied from logistic_regression_champion_20260324_214228 to models/latest/
